# Normalizing Toronto election results: municipal, provincial, federal

This is the first notebook in a rebuilt pipeline that ends with a latent-variable regression
model of poll-level turnout. Before anything else can be modelled or interpolated to census
tracts, the raw election results for the three most recent general elections touching
Toronto need to land in one common per-poll schema:

`electoral_district_number`, `electoral_district_name`, `polling_division_number`,
`polling_division_name`, `vote_type`, `number_of_votes`, `number_of_electors`,
`proportion_of_turnout`, `vote_in_other_division`, `source_note`, plus candidate/party vote
columns and (where available) polygon geometry.

The three raw sources could not be more different from each other:

- **Municipal 2023 mayoral race** — two Toronto Open Data Excel workbooks (a per-ward result
  workbook and a voter-statistics workbook) joined to a subdivision polygon atlas.
- **Provincial 2025 general election** — a single Elections Ontario "official return" CSV
  covering every candidate/poll row for the whole province, filtered down to Toronto's 25
  ridings, joined to election-atlas.ca polygon files per riding.
- **Federal 2025 general election** — Elections Canada's poll-by-poll "Format 2" CSV per
  riding (24 Toronto ridings), joined to election-atlas.ca polygon files per riding.

This notebook ports the core transformation logic from the archived script pipeline
(`analysis/archive/toronto_election_turnout/archive/elections/scripts/build_turnout_geojson.py` and
`build_candidate_party_votes.py`) rather than re-deriving the logic from scratch, because the
whole point of this stage is a faithful, verifiable normalization — not a redesign. The final
verification cell diffs the new output against the archived pipeline's committed CSVs
row-for-row and prints a PASS/FAIL summary.

**Explicitly out of scope for this notebook** (per the wider rebuild plan): the
`accessibility/` polling-place-distance analysis (a dead end never consumed downstream), the
Leaflet map viewers, and the `.mjs`/stakeholder-handoff packaging scripts. Those are not
reproduced anywhere in the new notebook pipeline.


In [1]:
import json
import math
import re
from collections import defaultdict
from pathlib import Path

import geopandas as gpd
import pandas as pd
from shapely.geometry import shape
from tqdm.notebook import tqdm


In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
REPO_ROOT = Path("../..").resolve()

RAW = REPO_ROOT / "data/toronto_election_turnout/archive/elections/raw"
SOURCE_DOWNLOADS = RAW / "source_downloads"

# Read-only ground truth from the archived pipeline, used only for verification at the end.
GROUND_TRUTH = REPO_ROOT / "data/toronto_election_turnout/archive/elections/processed"

# Fresh output tree for the rebuilt pipeline -- never overlaps with the archived outputs above.
OUT_DIR = REPO_ROOT / "data/toronto_election_turnout/elections"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Toronto's ridings/wards for each election ───────────────────────────────
# Federal electoral district numbers (45th general election) that touch the City of Toronto.
FEDERAL_CODES = [
    35007, 35022, 35023, 35024, 35026, 35029, 35030, 35031, 35041, 35092,
    35093, 35094, 35095, 35096, 35097, 35100, 35105, 35109, 35110, 35111,
    35112, 35117, 35120, 35122,
]

# Ontario provincial electoral district numbers (2025 general election) that touch Toronto.
PROVINCIAL_CODES = [
    7, 19, 20, 21, 22, 25, 28, 29, 30, 41, 83, 93, 94, 95, 96, 97, 98,
    101, 109, 110, 111, 112, 117, 120, 122,
]


## Shared parsing helpers

All three elections need the same small set of primitives: coercing messy numeric strings
(commas, blanks, NaN) to ints, computing a turnout ratio that is left `null` rather than a
nonsensical value above 1.0 when the source's vote/elector denominators are not coherent, and
building the human-readable `source_note` text that explains any special handling for a row.
Keeping these as shared functions (rather than re-implementing per election) is what makes the
three `vote_type` vocabularies and `source_note` conventions consistent across elections.


In [3]:
def nint(value):
    """Coerce to int, treating blank/NaN/unparseable as 0 (used for vote counts)."""
    if value is None or value == "" or (isinstance(value, float) and math.isnan(value)):
        return 0
    try:
        return int(float(str(value).replace(",", "")))
    except ValueError:
        return 0


def maybe_int(value):
    """Coerce to int, treating blank/NaN/unparseable as None (used for elector counts,
    where 'no data' and 'zero' are meaningfully different)."""
    if value is None or value == "" or (isinstance(value, float) and math.isnan(value)):
        return None
    try:
        return int(float(str(value).replace(",", "")))
    except ValueError:
        return None


def proportion(votes, electors):
    return None if votes is None or not electors else votes / electors


def turnout_ratio(votes, electors):
    """Turnout is left null (not clamped) when votes exceed electors -- see over_one_note."""
    value = proportion(votes, electors)
    return None if value is not None and value > 1 else value


def over_one_note(votes, electors):
    value = proportion(votes, electors)
    if value is not None and value > 1:
        return (
            "Official votes exceed official electors for this row; turnout ratio left null "
            "because the source denominator is not coherent for division-level turnout."
        )
    return None


def load_geojson(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def feature(properties, geometry):
    return {"type": "Feature", "properties": properties, "geometry": geometry}


def combine_notes(*notes):
    cleaned = [note for note in notes if note]
    return " ".join(cleaned) if cleaned else None


def unique_join(values):
    cleaned, seen = [], set()
    for value in values:
        if value is None or pd.isna(value):
            continue
        text = str(value).strip()
        if not text or text in seen:
            continue
        cleaned.append(text)
        seen.add(text)
    return "; ".join(cleaned) if cleaned else None


def clean_text(value):
    if value is None or pd.isna(value):
        return ""
    return str(value).strip()


## Municipal 2023 mayoral race

The municipal source data is two Toronto Open Data workbooks that don't share a key format:

- `toronto_2023_mayor.xlsx` has one sheet per ward, with subdivision numbers as column
  headers and one row per mayoral candidate -- this is where the actual vote counts live.
- `toronto_2023_mayor_voter_statistics.xlsx` has a flat `Ward`/`Sub` table with elector
  counts, building names, and rejected/declined ballot counts.

Both are joined to `toronto_2023_subdivisions.geojson` (the official ward-subdivision
polygon atlas) on a synthetic `WWSSS` code (2-digit ward + 3-digit subdivision).

Two things need care here, both handled the same way the archived script handled them:

1. **Subdivisions 96-99 are not ordinary polling places.** Toronto's voter-statistics
   workbook uses these as ward-level *reporting buckets*: 96 is long-term-care institutions,
   97 is mail-in voting, 98 and 99 are advance voting. They have vote totals but no polygon
   in the atlas (there's no single place to draw them), and mail-in/advance buckets have no
   elector denominator at all, so their `proportion_of_turnout` is left null with an
   explanatory `source_note` rather than guessed at.
2. **Municipal electors use the voter-statistics workbook's final `Total Eligible Electors`**
   (after list additions/deletions), which reproduces the officially reported ~37.2% turnout
   -- not the commonly-cited 38.5% figure, which uses an earlier, smaller denominator that
   would be wrong to substitute into subdivision-level rows.


In [4]:
def cleaned_municipal_ward_name(value):
    name = str(value).replace("City Ward ", "").strip()
    return re.sub(r"^\d+\s+", "", name)


def municipal_special_poll_name(poll):
    return {
        "096": "Long-term care combined reporting bucket",
        "097": "Mail-in voting reporting bucket",
        "098": "Advance vote reporting bucket",
        "099": "Advance vote reporting bucket",
    }.get(str(poll).zfill(3))


def municipal_vote_type(poll):
    poll = str(poll).zfill(3)
    if poll == "097":
        return "mail_in"
    if poll in {"098", "099"}:
        return "advance"
    if poll == "096":
        return "special"
    return "election_day"


def municipal_regular_note(votes, electors, code_, in_workbook):
    details = []
    if not in_workbook:
        if electors is None:
            details.append(
                "Official subdivision geometry exists, but the official mayoral result workbook has no vote count "
                "and the official voter-statistics workbook has no elector count."
            )
        else:
            details.append(
                f"Official voter-statistics workbook reports {electors} electors, but the official mayoral result workbook has no vote count."
            )
    if in_workbook and electors is None:
        details.append(
            f"The official mayoral result workbook reports {votes} votes, but the official voter-statistics workbook has no elector count."
        )
    if votes is not None and electors is not None and votes > electors:
        details.append(
            f"The official mayoral result workbook reports {votes} votes, while the official voter-statistics workbook "
            f"reports {electors} electors. The original counts are retained, but turnout is left blank because the "
            "denominator is not coherent for division-level turnout."
        )
    if details:
        details.append("No explicit source join to or from another ordinary subdivision was found.")
    return " ".join(details) if details else None


def municipal_special_note(code_, votes):
    ward = str(int(code_[:2]))
    poll = code_[2:]
    origin = {"096": "long-term care institutions", "097": "mail-in voting"}.get(poll, "advance voting")
    return (
        f"Subdivision {int(poll)} in ward {ward} is a special ward-level reporting bucket for {origin}, "
        f"not an ordinary mapped voting subdivision. The official mayoral result workbook reports {votes} votes in this bucket. "
        "The subdivision geometry source has no polygon for this bucket. "
        f"Join in: {votes} votes from {origin}. "
        "Join out: none indicated. Contributing ordinary subdivision numbers are not identified in the source."
    )


def build_municipal():
    geom_data = load_geojson(RAW / "toronto_2023_subdivisions.geojson")
    geom = {}
    for feat in geom_data["features"]:
        code_ = str(feat["properties"]["AREA_LONG_CODE"]).zfill(5)
        geom[code_] = feat["geometry"]

    stats = pd.read_excel(RAW / "toronto_2023_mayor_voter_statistics.xlsx", sheet_name="2023 Voter Turnout Statisti M")
    stats_by_code = {}
    for _, row in stats.iterrows():
        ward = maybe_int(row.get("Ward"))
        sub = maybe_int(row.get("Sub"))
        if ward is None or sub is None:
            continue
        code_ = f"{ward:02d}{sub:03d}"
        stats_by_code[code_] = {
            "polling_division_name": str(row.get("Building Name")).strip() if not pd.isna(row.get("Building Name")) else None,
            "number_of_votes": maybe_int(row.get("Number Voted")),
            "number_of_electors": maybe_int(row.get("Total Eligible Electors")),
        }

    xls = pd.ExcelFile(RAW / "toronto_2023_mayor.xlsx")
    ward_sheets = [s for s in xls.sheet_names if s.startswith("Ward ")]
    votes_by_code = defaultdict(int)
    ward_names = {}
    for sheet in tqdm(ward_sheets, desc="Municipal wards"):
        ward = int(sheet.split()[1])
        df = pd.read_excel(xls, sheet_name=sheet, header=None)
        ward_names[str(ward).zfill(2)] = cleaned_municipal_ward_name(str(df.iloc[0, 0]))
        subdivisions = [v for v in list(df.iloc[1, 1:]) if str(v) != "nan" and str(v) != "Total"]
        total_row = next(
            (idx for idx, value in df.iloc[:, 0].items() if str(value).strip().startswith(f"City Ward {ward} Totals")),
            len(df),
        )
        for col_offset, sub in enumerate(subdivisions, start=1):
            code_ = f"{ward:02d}{str(nint(sub)).zfill(3)}"
            votes_by_code[code_] += sum(nint(value) for value in df.iloc[3:total_row, col_offset])

    features = []
    for code_, geometry in sorted(geom.items()):
        ward = code_[:2]
        stat = stats_by_code.get(code_, {})
        votes = votes_by_code.get(code_, stat.get("number_of_votes"))
        electors = stat.get("number_of_electors")
        props = {
            "electoral_district_number": ward,
            "electoral_district_name": ward_names.get(ward),
            "polling_division_number": str(int(code_[2:])),
            "polling_division_name": stat.get("polling_division_name"),
            "vote_type": municipal_vote_type(code_[2:]),
            "number_of_votes": votes,
            "number_of_electors": electors,
            "proportion_of_turnout": turnout_ratio(votes, electors),
            "vote_in_other_division": None,
            "source_note": municipal_regular_note(votes, electors, code_, code_ in votes_by_code),
        }
        features.append(feature(props, geometry))
    for code_, votes in sorted(votes_by_code.items()):
        if code_ in geom:
            continue
        ward, poll = code_[:2], code_[2:]
        stat = stats_by_code.get(code_, {})
        votes = stat.get("number_of_votes", votes)
        electors = stat.get("number_of_electors")
        props = {
            "electoral_district_number": ward,
            "electoral_district_name": ward_names.get(ward),
            "polling_division_number": str(int(poll)),
            "polling_division_name": stat.get("polling_division_name") or municipal_special_poll_name(poll),
            "vote_type": municipal_vote_type(poll),
            "number_of_votes": votes,
            "number_of_electors": electors,
            "proportion_of_turnout": turnout_ratio(votes, electors),
            "vote_in_other_division": None,
            "source_note": municipal_special_note(code_, votes),
        }
        features.append(feature(props, None))
    return features


municipal_features = build_municipal()
print(f"Municipal 2023: {len(municipal_features)} subdivision rows, "
      f"{sum(1 for f in municipal_features if f['geometry'])} with geometry.")


Municipal wards:   0%|          | 0/25 [00:00<?, ?it/s]

Municipal 2023: 1545 subdivision rows, 1445 with geometry.


## Provincial 2025 general election

Elections Ontario publishes one giant "official return" CSV for the whole province
(`eo_2025_official_return.csv`); it is filtered down to Toronto's 25 ridings by matching the
3-digit district-number prefix on `ElectoralDistrictNameEnglish`.

Two normalization steps are specific to this source:

1. **Suffix polls are aggregated to the base poll number that the atlas polygon actually
   represents.** Elections Ontario sometimes reports the same physical poll as several rows
   with a letter/number suffix (e.g. two rows for poll 12 split by ballot box); these are
   summed into one `polling_division_number` so that every row matches exactly one polygon.
2. **Combined polls are flagged, not merged away.** When Elections Ontario marks a poll as
   combined with another for reporting purposes, both the target and the contributing rows
   are kept, each carrying a `source_note` and (for the contributing row) a
   `vote_in_other_division` pointer -- so a reader can tell "no data" apart from "the data is
   reported elsewhere."

`ADV...`-style advance-vote rows have no per-poll polygon in the atlas at all and often no
elector denominator; they are kept as rows with null geometry rather than dropped, so their
vote counts are not silently lost from riding totals.


In [5]:
def provincial_vote_type(value):
    return "advance" if str(value or "").strip().lower() == "advance" else "election_day"


def provincial_geometry_index():
    index, attrs = {}, {}
    for code_ in tqdm(PROVINCIAL_CODES, desc="Provincial riding polygons"):
        data = load_geojson(SOURCE_DOWNLOADS / "provincial_polygons" / f"{code_}.geojson")
        for feat in data["features"]:
            props = feat["properties"]
            poll = str(props.get("POLLNO") or props.get("PD_LABEL") or "").strip()
            if poll:
                index[(str(code_).zfill(3), poll.zfill(3))] = feat["geometry"]
                attrs[(str(code_).zfill(3), poll.zfill(3))] = props
    return index, attrs


def build_provincial():
    geom, geom_attrs = provincial_geometry_index()
    df = pd.read_csv(RAW / "eo_2025_official_return.csv", dtype=str)
    df = df[df["EventNameEnglish"].str.contains("2025 Provincial General Election", na=False)]
    wanted = {str(c).zfill(3) for c in PROVINCIAL_CODES}
    df["district_number"] = df["ElectoralDistrictNameEnglish"].str.extract(r"^(\d{3})")
    df = df[df["district_number"].isin(wanted)]

    df["base_poll"] = df["PollNumber"].str.extract(r"^(\d+)", expand=False)
    advance_df = df[df["base_poll"].isna()].copy()
    df = df[df["base_poll"].notna()]

    grouped = {}
    for (district, district_name, base_poll), g in df.groupby(
        ["district_number", "ElectoralDistrictNameEnglish", "base_poll"], dropna=False
    ):
        raw_polls = sorted(str(v) for v in g["PollNumber"].dropna().unique())
        electors = rejected = unmarked = declined = 0
        for _, poll_rows in g.groupby("PollNumber", dropna=False):
            electors += nint(poll_rows["NamesOnListOfElectors"].dropna().iloc[0]) if len(poll_rows) else 0
            rejected += nint(poll_rows["BallotsFromBoxesRejectedAsMarkings"].dropna().iloc[0]) if len(poll_rows) else 0
            unmarked += nint(poll_rows["BallotsFromBoxesUnmarkedByVoters"].dropna().iloc[0]) if len(poll_rows) else 0
            declined += nint(poll_rows["BallotsDeclinedByVoters"].dropna().iloc[0]) if len(poll_rows) else 0

        votes = sum(nint(v) for v in g["AcceptedBallotCount"])
        combined_with = next((str(v) for v in g["CombinedWith"].dropna() if str(v).strip()), None)
        if combined_with:
            m = re.match(r"^(\d+)", combined_with)
            combined_with = m.group(1).zfill(3) if m else combined_with
        poll_s = str(base_poll).zfill(3)
        suffix_note = f"Aggregated official suffix polls: {', '.join(raw_polls)}." if len(raw_polls) > 1 else None
        combined_note = None
        if combined_with and combined_with != poll_s:
            combined_note = f"Combined official poll; results are reported with division {combined_with}."
        elif combined_with == poll_s:
            combined_note = "Official combined reporting target for one or more divisions."

        grouped[(district, poll_s)] = {
            "electoral_district_number": district,
            "electoral_district_name": re.sub(r"^\d{3}\s+", "", district_name),
            "polling_division_number": poll_s,
            "polling_division_name": unique_join(g["VotingPlaceAddressOrLocation"]),
            "vote_type": provincial_vote_type(unique_join(g["PollCategory"])),
            "number_of_votes": votes + rejected + unmarked + declined,
            "number_of_electors": electors,
            "proportion_of_turnout": turnout_ratio(votes + rejected + unmarked + declined, electors),
            "vote_in_other_division": combined_with if combined_with and combined_with != poll_s else None,
            "source_note": combine_notes(
                suffix_note, combined_note,
                over_one_note(votes + rejected + unmarked + declined, electors),
            ),
        }

    features = []
    emitted_source_keys = set()
    for key, props in sorted(grouped.items()):
        geometry = geom.get(key)
        if geometry is not None:
            emitted_source_keys.add(key)
        features.append(feature(props, geometry))

    for (district, district_name, poll), g in advance_df.groupby(
        ["district_number", "ElectoralDistrictNameEnglish", "PollNumber"], dropna=False
    ):
        electors = nint(g["NamesOnListOfElectors"].dropna().iloc[0]) if len(g) else 0
        electors = electors or None
        rejected = nint(g["BallotsFromBoxesRejectedAsMarkings"].dropna().iloc[0]) if len(g) else 0
        unmarked = nint(g["BallotsFromBoxesUnmarkedByVoters"].dropna().iloc[0]) if len(g) else 0
        declined = nint(g["BallotsDeclinedByVoters"].dropna().iloc[0]) if len(g) else 0
        votes = sum(nint(v) for v in g["AcceptedBallotCount"]) + rejected + unmarked + declined
        if not votes:
            continue
        props = {
            "electoral_district_number": district,
            "electoral_district_name": re.sub(r"^\d{3}\s+", "", district_name),
            "polling_division_number": str(poll),
            "polling_division_name": unique_join(g["VotingPlaceAddressOrLocation"]),
            "vote_type": "advance",
            "number_of_votes": votes,
            "number_of_electors": electors,
            "proportion_of_turnout": turnout_ratio(votes, electors),
            "vote_in_other_division": None,
            "source_note": combine_notes(
                ("Official advance-vote reporting bucket; no ordinary polling-division polygon is provided for this row. "
                 "Votes are included in the riding total."),
                ("No separate elector count is provided for this advance-vote row, so turnout is left blank."
                 if electors is None else None),
                over_one_note(votes, electors),
            ),
        }
        features.append(feature(props, None))

    for key, geometry in sorted(geom.items()):
        if key in emitted_source_keys:
            continue
        attrs = geom_attrs[key]
        props = {
            "electoral_district_number": key[0],
            "electoral_district_name": attrs.get("RIDINGNAME"),
            "polling_division_number": attrs.get("PD_LABEL") or str(attrs.get("POLLNO")),
            "polling_division_name": None,
            "vote_type": "election_day",
            "number_of_votes": nint(attrs.get("TOTALVOTES")),
            "number_of_electors": None,
            "proportion_of_turnout": None,
            "vote_in_other_division": attrs.get("COMBINED") or attrs.get("NOTE"),
            "source_note": "Atlas polygon had vote totals, but no matching official electors row was found; turnout left null.",
        }
        features.append(feature(props, geometry))
    return features


provincial_features = build_provincial()
print(f"Provincial 2025: {len(provincial_features)} poll rows, "
      f"{sum(1 for f in provincial_features if f['geometry'])} with geometry.")


Provincial riding polygons:   0%|          | 0/25 [00:00<?, ?it/s]

Provincial 2025: 1532 poll rows, 1388 with geometry.


## Federal 2025 general election

Elections Canada's poll-by-poll "Format 2" CSV (one file per riding, 24 Toronto ridings) is
the most structurally complex of the three sources, because of **combined polls**: Elections
Canada sometimes decides two physically distinct polls should report as a single result --
usually because turnout at one location was too low to report separately. The source marks
the *contributing* poll's row with a `Combined with No.` pointing at the *target* poll.

The archived pipeline's method, ported unchanged here, is:

- The contributing poll's **electors are added to the target poll's elector count** (so the
  city-wide denominator is conserved -- nobody's elector registration disappears).
- The contributing poll's **votes are not double-counted**: they were already tabulated
  under the target poll by Elections Canada, so the contributing row's own
  `number_of_votes`/`proportion_of_turnout` are left null.
- The contributing row is still emitted (with its own polygon, if the atlas has one) so it
  remains visible on a map, carrying `vote_in_other_division` pointing at the target poll and
  a `source_note` explaining the elector transfer.

This matters downstream: a later notebook that interpolates poll results to census tracts by
area-weighting polygons needs every polygon accounted for exactly once with a coherent
elector count -- a combined poll's polygon should absorb its share of the *electorate*
without also claiming votes that are already attributed to the target polygon.

A second wrinkle is **poll-number suffixes**: Format 2 sometimes reports `12A`/`12B` style
sub-polls or `S/R 4`-style special-voting-rules ("SVR") groups; these are canonicalized to
match the atlas polygon's own numbering (`12`, `SVR-4`) and their vote/elector counts are
summed under that canonical number.


In [6]:
def cleaned_federal_poll_name(value):
    if value is None or pd.isna(value):
        return None
    name = str(value).strip()
    return name or None


def federal_vote_type(poll, poll_name=None):
    poll_text = str(poll or "").strip()
    name = str(poll_name or "").lower()
    if poll_text.startswith("SVR") or poll_text.startswith("S/R"):
        return "special"
    if poll_text.isdigit() and int(poll_text) >= 600:
        return "advance"
    if "group" in name or "groupe" in name:
        return "special"
    return "election_day"


def canonical_federal_poll(value):
    if value is None or pd.isna(value):
        return ""
    text = str(value or "").strip()
    special_match = re.match(r"^S/R\s*(\d+)$", text, re.IGNORECASE)
    if special_match:
        return f"SVR-{special_match.group(1)}"
    letter_match = re.match(r"^(\d+)[A-Za-z]$", text)
    if letter_match:
        return letter_match.group(1)
    return text


def federal_poll_sort_value(value):
    text = str(value)
    if text.startswith("SVR-"):
        return 999998, int(text.split("-", 1)[1]), ""
    m = re.match(r"^(\d+)(?:-(\d+))?$", text)
    if not m:
        return 999999, 0, text
    return int(m.group(1)), int(m.group(2) or 0), text


def federal_geometry_index():
    index, attrs = {}, {}
    for code_ in tqdm(FEDERAL_CODES, desc="Federal riding polygons"):
        data = load_geojson(SOURCE_DOWNLOADS / "federal_polygons" / f"{code_}.geojson")
        for feat in data["features"]:
            props = feat["properties"]
            poll = str(props.get("EMRP_NAME") or props.get("PD_NUM") or "").strip()
            if poll:
                index[(str(code_), poll)] = feat["geometry"]
                attrs[(str(code_), poll)] = props
    return index, attrs


FEDERAL_COLS = dict(
    district_name="Electoral District Name_English/Nom de circonscription_Anglais",
    district_number="Electoral District Number/Num\u00e9ro de circonscription",
    poll="Polling Division Number/Num\u00e9ro de section de vote",
    poll_name="Polling Division Name/Nom de section de vote",
    combined="Combined with No./R\u00e9sultats combin\u00e9s \u00e0 ceux du n\u00b0",
    rejected="Rejected Ballots for poll/Bulletins rejet\u00e9s du bureau",
    electors="Electors for poll/\u00c9lecteurs du bureau",
    candidate_vote="Candidate Vote Count/Votes du candidat",
    void="Void Poll Indicator/Indicateur de bureau supprim\u00e9",
    no_poll="No Poll Held Indicator/Indicateur de bureau sans scrutin",
)


def build_federal():
    geom, geom_attrs = federal_geometry_index()
    features = []
    emitted_source_keys = set()
    c = FEDERAL_COLS

    for code_ in tqdm(FEDERAL_CODES, desc="Federal ridings"):
        df = pd.read_csv(SOURCE_DOWNLOADS / "federal_csv_format2" / f"{code_}.csv", encoding="utf-8-sig", dtype=str)

        records, combined_records = {}, []
        elector_additions = defaultdict(int)
        elector_addition_sources = defaultdict(list)
        for raw_poll, group in df.groupby(c["poll"], dropna=False, sort=False):
            source_poll = str(raw_poll or "").strip()
            if not source_poll:
                continue
            poll = canonical_federal_poll(source_poll)
            first = group.iloc[0]
            combined_target = canonical_federal_poll(first.get(c["combined"]))
            if combined_target:
                combined_electors = maybe_int(first[c["electors"]])
                if combined_electors is not None:
                    elector_additions[combined_target] += combined_electors
                elector_addition_sources[combined_target].append(source_poll)
                combined_records.append((poll, source_poll, combined_target, first))
                continue
            record = records.setdefault(poll, {
                "row": first, "votes": 0, "electors": 0, "electors_known": False,
                "source_polls": [], "void": False, "no_poll": False,
            })
            record["votes"] += sum(nint(v) for v in group[c["candidate_vote"]]) + nint(first[c["rejected"]])
            electors_value = maybe_int(first[c["electors"]])
            if electors_value is not None:
                record["electors"] += electors_value
                record["electors_known"] = True
            record["source_polls"].append(source_poll)
            record["void"] = record["void"] or str(first.get(c["void"], "")).strip().upper() == "Y"
            record["no_poll"] = record["no_poll"] or str(first.get(c["no_poll"], "")).strip().upper() == "Y"

        for poll, record in sorted(records.items(), key=lambda item: federal_poll_sort_value(item[0])):
            row = record["row"]
            votes = record["votes"]
            electors = record["electors"] if record["electors_known"] else None
            if elector_additions[poll]:
                electors = (electors or 0) + elector_additions[poll]
            zero_elector_bucket = electors == 0 and votes > 0
            if zero_elector_bucket:
                electors = None
            geometry = geom.get((str(code_), poll))
            if geometry is not None:
                emitted_source_keys.add((str(code_), poll))
            source_polls = record["source_polls"]
            if poll.startswith("SVR-") and source_polls != [poll]:
                aggregated_note = f"Official source label normalized to {poll}: {', '.join(source_polls)}."
            elif source_polls != [poll]:
                aggregated_note = f"Official subpoll rows aggregated to match the atlas polygon: {', '.join(source_polls)}."
            else:
                aggregated_note = None
            no_geometry_note = (
                "No ordinary polling-division polygon was found in the atlas geometry source." if geometry is None else None
            )
            combined_target_note = (
                f"Electors from combined divisions added here: {', '.join(elector_addition_sources[poll])}."
                if elector_addition_sources[poll] else None
            )
            status_note = combine_notes(
                "Official source marks this as a void poll." if record["void"] else None,
                "Official source marks this as a no-poll-held row." if record["no_poll"] else None,
            )
            zero_elector_note = (
                "Elections Canada Format 2 reports 0 electors for this reporting bucket; treated as no supported elector denominator."
                if zero_elector_bucket else None
            )
            props = {
                "electoral_district_number": str(code_),
                "electoral_district_name": str(row[c["district_name"]]),
                "polling_division_number": poll,
                "polling_division_name": cleaned_federal_poll_name(row[c["poll_name"]]),
                "vote_type": federal_vote_type(poll, row[c["poll_name"]]),
                "number_of_votes": votes,
                "number_of_electors": electors,
                "proportion_of_turnout": turnout_ratio(votes, electors),
                "vote_in_other_division": None,
                "source_note": combine_notes(
                    aggregated_note, combined_target_note, status_note, zero_elector_note, no_geometry_note,
                    over_one_note(votes, electors),
                ),
            }
            features.append(feature(props, geometry))

        for poll, source_poll, combined_target, row in combined_records:
            source_key = (str(code_), source_poll)
            geometry = None if source_key in emitted_source_keys else geom.get(source_key)
            if geometry is not None:
                emitted_source_keys.add(source_key)
            props = {
                "electoral_district_number": str(code_),
                "electoral_district_name": str(row[c["district_name"]]),
                "polling_division_number": source_poll,
                "polling_division_name": cleaned_federal_poll_name(row[c["poll_name"]]),
                "vote_type": federal_vote_type(source_poll, row[c["poll_name"]]),
                "number_of_votes": None,
                "number_of_electors": maybe_int(row[c["electors"]]),
                "proportion_of_turnout": None,
                "vote_in_other_division": combined_target,
                "source_note": (
                    f"Combined official row {source_poll}; electors are added to division {combined_target}, "
                    "and votes are reported with that target division."
                ),
            }
            features.append(feature(props, geometry))

    def source_sort_key(item):
        code_, poll = item[0]
        m = re.match(r"(\d+)", str(poll))
        return code_, int(m.group(1)) if m else 999999, str(poll)

    for (code_, poll), geometry in sorted(geom.items(), key=source_sort_key):
        if (code_, poll) in emitted_source_keys:
            continue
        attrs = geom_attrs[(code_, poll)]
        props = {
            "electoral_district_number": code_,
            "electoral_district_name": attrs.get("RIDINGNAME"),
            "polling_division_number": poll,
            "polling_division_name": None,
            "vote_type": federal_vote_type(poll),
            "number_of_votes": nint(attrs.get("TOTALVOTES")),
            "number_of_electors": None,
            "proportion_of_turnout": None,
            "vote_in_other_division": attrs.get("MERGED_WIT"),
            "source_note": "Atlas polygon had vote totals, but no matching official electors row was found; turnout left null.",
        }
        features.append(feature(props, geometry))
    return features


federal_features = build_federal()
print(f"Federal 2025: {len(federal_features)} poll rows, "
      f"{sum(1 for f in federal_features if f['geometry'])} with geometry.")


Federal riding polygons:   0%|          | 0/24 [00:00<?, ?it/s]

Federal ridings:   0%|          | 0/24 [00:00<?, ?it/s]

Federal 2025: 5069 poll rows, 4273 with geometry.


## Candidate and party vote columns

Each election also needs a per-poll breakdown by candidate and party, so the turnout table
alone is not the whole story: a poll's *composition* of the vote (e.g. margin between top two
candidates) is a Block-4 competitiveness feature used later in the pipeline. Rather than
carry a separate long candidate table forward, this notebook aggregates straight to wide
`party_<slug>_votes` columns on the poll table -- there are at most 15 parties in any single
election, so a wide table is small and easy to join.

The three sources need three different strategies to recover `party_name` for each row:

- **Municipal** is non-partisan by law, so every candidate is tagged `"Non-partisan"` and
  there is exactly one party column (`party_non_partisan_votes`, which is identical to
  `number_of_votes`/`poll_total_candidate_votes` for that reason).
- **Federal** Format 2 rows carry the candidate's party affiliation directly
  (`Political Affiliation Name_English`).
- **Provincial**'s official return has candidate names and votes but *no party column at
  all*. The party has to be joined in from two other Elections Ontario files: a
  candidate-summary CSV (candidate's *riding-wide* total votes) and a political-interest-code
  lookup, matched on `(district, candidate's total votes)` -- a slightly indirect join, but it
  is the only key the two files share.

Federal **combined** rows are excluded from the candidate tally here (their votes are already
counted under the target poll in the Format 2 source), which is why `poll_total_candidate_votes`
for a combined-target poll already includes the contributing poll's candidate votes even
though the contributing row's own `number_of_votes` is null.


In [7]:
def slug(value):
    import unicodedata
    text = unicodedata.normalize("NFKD", str(value))
    text = text.encode("ascii", "ignore").decode("ascii").lower()
    return re.sub(r"[^a-z0-9]+", "_", text).strip("_")


def poll_id(election_id, district, poll, vote_type):
    poll_value = clean_text(poll)
    if poll_value.isdigit():
        poll_value = str(int(poll_value))
    return "|".join([election_id, clean_text(district), poll_value, clean_text(vote_type)])


def party_columns(rows):
    parties = sorted({row["party_name"] for row in rows if row["party_name"]})
    return {party: f"party_{slug(party)}_votes" for party in parties}


def poll_candidate_aggregates(election_id, rows):
    columns = party_columns(rows)
    aggregates = {}
    for row in rows:
        pid = poll_id(election_id, row["electoral_district_number"], row["polling_division_number"], row["vote_type"])
        if pid not in aggregates:
            aggregates[pid] = {"poll_total_candidate_votes": 0, **{c: 0 for c in columns.values()}}
        aggregates[pid]["poll_total_candidate_votes"] += row["candidate_vote_count"]
        aggregates[pid][columns[row["party_name"]]] += row["candidate_vote_count"]
    return columns, aggregates


def load_turnout_context(election_id, features):
    """district/poll -> turnout-row context, keyed the same way the candidate rows will be."""
    context = {}
    for f in features:
        row = f["properties"]
        district = clean_text(row["electoral_district_number"])
        if election_id == "provincial_2025":
            district = district.zfill(3)
        elif election_id == "municipal_2023_mayor":
            district = district.zfill(2)
        poll = clean_text(row["polling_division_number"])
        context[(district, poll)] = {
            "polling_division_name": clean_text(row.get("polling_division_name")) or None,
            "vote_type": clean_text(row.get("vote_type")) or None,
            "vote_in_other_division": clean_text(row.get("vote_in_other_division")) or None,
        }
    return context


def municipal_candidate_rows(features):
    context = load_turnout_context("municipal_2023_mayor", features)
    xls = pd.ExcelFile(RAW / "toronto_2023_mayor.xlsx")
    rows = []
    for sheet in xls.sheet_names:
        if not sheet.startswith("Ward "):
            continue
        ward = str(int(sheet.split()[1])).zfill(2)
        df = pd.read_excel(xls, sheet_name=sheet, header=None)
        subdivisions = [str(nint(v)) for v in df.iloc[1, 1:].tolist() if not pd.isna(v) and str(v).strip() != "Total"]
        total_row = next(
            (idx for idx, value in df.iloc[:, 0].items() if str(value).strip().startswith(f"City Ward {int(ward)} Totals")),
            len(df),
        )
        for col_offset, poll in enumerate(subdivisions, start=1):
            poll_rows = []
            for row_idx in range(3, total_row):
                candidate = clean_text(df.iloc[row_idx, 0])
                if not candidate:
                    continue
                poll_rows.append((candidate, nint(df.iloc[row_idx, col_offset])))
            ctx = context.get((ward, poll), {})
            for candidate, votes in poll_rows:
                rows.append({
                    "electoral_district_number": ward, "polling_division_number": poll,
                    "vote_type": ctx.get("vote_type") or ("election_day" if poll not in {"96", "97", "98", "99"} else None),
                    "candidate_name": candidate, "party_name": "Non-partisan", "candidate_vote_count": votes,
                })
    return rows


def provincial_candidate_rows(features):
    context = load_turnout_context("provincial_2025", features)
    wanted = {str(code_).zfill(3) for code_ in PROVINCIAL_CODES}
    df = pd.read_csv(RAW / "eo_2025_official_return.csv", dtype=str)
    df = df[df["EventNameEnglish"].str.contains("2025 Provincial General Election", na=False)]
    df["district_number"] = df["ElectoralDistrictNameEnglish"].str.extract(r"^(\d{3})")
    df = df[df["district_number"].isin(wanted)].copy()
    df["base_poll"] = df["PollNumber"].str.extract(r"^(\d+)", expand=False)
    df["output_poll"] = df["base_poll"].where(df["base_poll"].notna(), df["PollNumber"])
    df["output_poll"] = df["output_poll"].map(lambda v: str(v).zfill(3) if str(v).isdigit() else str(v))

    candidate_summary = pd.read_csv(SOURCE_DOWNLOADS / "eo_2025_candidate_summary.csv", dtype=str, encoding="utf-8-sig")
    candidate_summary = candidate_summary[candidate_summary["EventNameEnglish"].eq("2025 Provincial General Election")].copy()
    candidate_summary["district_number"] = candidate_summary["ElectoralDistrictNumber"].str.zfill(3)
    candidate_summary = candidate_summary[candidate_summary["district_number"].isin(wanted)]
    candidate_summary["candidate_total_votes"] = candidate_summary["TotalValidBallotsCast"].map(nint)

    party_codes = pd.read_csv(SOURCE_DOWNLOADS / "eo_2025_political_interest_codes.csv", dtype=str, encoding="utf-8-sig")
    party_codes = party_codes[party_codes["EventNameEnglish"].eq("2025 Provincial General Election")][
        ["PoliticalInterestCode", "PartyFullNameEnglish"]].drop_duplicates()
    party_lookup = (
        candidate_summary.merge(party_codes, on="PoliticalInterestCode", how="left")
        .set_index(["district_number", "candidate_total_votes"])["PartyFullNameEnglish"].to_dict()
    )

    candidate_totals = df.assign(candidate_votes=df["AcceptedBallotCount"].map(nint)).groupby(
        ["district_number", "NameOfCandidates"])["candidate_votes"].sum().to_dict()

    rows = []
    for (district, poll, candidate), g in df.groupby(["district_number", "output_poll", "NameOfCandidates"], dropna=False):
        candidate = clean_text(candidate)
        if not candidate:
            continue
        votes = sum(nint(v) for v in g["AcceptedBallotCount"])
        candidate_total_votes = candidate_totals[(district, candidate)]
        party_name = party_lookup.get((district, candidate_total_votes))
        if not party_name:
            raise ValueError(f"No official party match for provincial candidate {candidate!r} in district {district}")
        poll_for_context = str(int(poll)) if str(poll).isdigit() else str(poll)
        rows.append({
            "electoral_district_number": district, "polling_division_number": poll_for_context,
            "vote_type": context.get((district, poll_for_context), {}).get("vote_type") or ("advance" if str(poll).startswith("ADV") else "election_day"),
            "candidate_name": candidate, "party_name": party_name, "candidate_vote_count": votes,
        })
    return rows


def federal_candidate_rows(features):
    context = load_turnout_context("federal_2025", features)
    frames = [pd.read_csv(SOURCE_DOWNLOADS / "federal_csv_format2" / f"{code_}.csv", encoding="utf-8-sig", dtype=str)
              for code_ in FEDERAL_CODES]
    df = pd.concat(frames, ignore_index=True)
    c = FEDERAL_COLS
    family_col = "Candidate\u2019s Family Name/Nom de famille du candidat"
    middle_col = "Candidate\u2019s Middle Name/Second pr\u00e9nom du candidat"
    first_col = "Candidate\u2019s First Name/Pr\u00e9nom du candidat"
    party_col = "Political Affiliation Name_English/Appartenance politique_Anglais"

    df["output_poll"] = df[c["poll"]].map(canonical_federal_poll)
    df = df[df[c["combined"]].fillna("").str.strip().eq("")].copy()  # combined rows already counted under target
    df["candidate_name"] = df.apply(
        lambda row: " ".join(p for p in [clean_text(row[first_col]), clean_text(row[middle_col]), clean_text(row[family_col])] if p),
        axis=1,
    )

    rows = []
    for (district, poll, candidate, party), g in df.groupby(
        [c["district_number"], "output_poll", "candidate_name", party_col], dropna=False
    ):
        district = clean_text(district)
        poll = clean_text(poll)
        if not candidate:
            continue
        rows.append({
            "electoral_district_number": district, "polling_division_number": poll,
            "vote_type": context.get((district, poll), {}).get("vote_type") or federal_vote_type(poll),
            "candidate_name": candidate, "party_name": clean_text(party) or None,
            "candidate_vote_count": int(sum(nint(v) for v in g[c["candidate_vote"]])),
        })
    return rows


municipal_candidates = municipal_candidate_rows(municipal_features)
provincial_candidates = provincial_candidate_rows(provincial_features)
federal_candidates = federal_candidate_rows(federal_features)
print(f"Candidate-vote rows -- municipal: {len(municipal_candidates)}, "
      f"provincial: {len(provincial_candidates)}, federal: {len(federal_candidates)}")


Candidate-vote rows -- municipal: 148002, provincial: 8874, federal: 24394


## Assembling one poll-level table per election

Each election's turnout rows and candidate/party rows are now joined on the same `poll_id`
key (`"<election_id>|<district>|<poll>|<vote_type>"`) into a single wide table: turnout
columns, then `poll_total_candidate_votes`, then one column per party, then
`vote_in_other_division`/`source_note`. This differs slightly from the archived pipeline,
which split turnout and candidate-details into separate CSVs per election joined by
`poll_id` plus a normalized long candidate table -- that split existed mainly to support the
old Leaflet viewer's incremental loading, which is out of scope here. One table per election
is simpler for a notebook reader and for the next notebook in the pipeline to consume.

`electoral_district_name` is also kept inline (the archived pipeline moved it to a separate
`_districts.csv` lookup file to shave a few bytes off a viewer payload) since a single
consolidated file, not a family of lookup files, is the goal here.

Geometry is written to a **separate GeoJSON file**, not embedded as a JSON-string column in
the CSV: the CSV stays lean and easy to inspect, while the GeoJSON keeps every polygon (and a
`null` geometry placeholder for rows with no atlas polygon) keyed by the same `poll_id`, ready
for the population-weighted poll-to-census-tract interpolation notebook that follows this one.


In [8]:
def assemble_election(election_id, features, candidate_rows, out_stem):
    columns, aggregates = poll_candidate_aggregates(election_id, candidate_rows)
    party_fields = list(columns.values())

    poll_df = pd.DataFrame([f["properties"] for f in features])
    geometries = [f["geometry"] for f in features]
    poll_df["poll_id"] = poll_df.apply(
        lambda r: poll_id(election_id, r["electoral_district_number"], r["polling_division_number"], r["vote_type"]),
        axis=1,
    )
    poll_df["poll_total_candidate_votes"] = poll_df["poll_id"].map(
        lambda pid: aggregates.get(pid, {}).get("poll_total_candidate_votes")
    )
    for field in party_fields:
        poll_df[field] = poll_df["poll_id"].map(lambda pid, field=field: aggregates.get(pid, {}).get(field))

    ordered = [
        "poll_id", "electoral_district_number", "electoral_district_name",
        "polling_division_number", "polling_division_name", "vote_type",
        "number_of_votes", "poll_total_candidate_votes", "number_of_electors",
        "proportion_of_turnout", *party_fields, "vote_in_other_division", "source_note",
    ]
    poll_df = poll_df[ordered]

    poll_df.to_csv(OUT_DIR / f"{out_stem}_polls.csv", index=False)

    shapes = [shape(g) if g else None for g in geometries]
    gdf = gpd.GeoDataFrame(poll_df.copy(), geometry=shapes, crs="EPSG:4326")
    gdf.to_file(OUT_DIR / f"{out_stem}_polls.geojson", driver="GeoJSON")

    return poll_df, party_fields


municipal_df, municipal_party_fields = assemble_election(
    "municipal_2023_mayor", municipal_features, municipal_candidates, "municipal_2023_mayor"
)
provincial_df, provincial_party_fields = assemble_election(
    "provincial_2025", provincial_features, provincial_candidates, "provincial_2025"
)
federal_df, federal_party_fields = assemble_election(
    "federal_2025", federal_features, federal_candidates, "federal_2025"
)

for name, df in [("municipal_2023_mayor", municipal_df), ("provincial_2025", provincial_df), ("federal_2025", federal_df)]:
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns -> {OUT_DIR / (name + '_polls.csv')}")


municipal_2023_mayor: 1545 rows, 13 columns -> /home/aniket/Programming/place-and-politics-toronto/data/toronto_election_turnout/elections/municipal_2023_mayor_polls.csv
provincial_2025: 1532 rows, 27 columns -> /home/aniket/Programming/place-and-politics-toronto/data/toronto_election_turnout/elections/provincial_2025_polls.csv
federal_2025: 5069 rows, 25 columns -> /home/aniket/Programming/place-and-politics-toronto/data/toronto_election_turnout/elections/federal_2025_polls.csv


## Verification against the archived pipeline's output

The archived pipeline's committed CSVs under
`data/toronto_election_turnout/archive/elections/processed/<election>/` are the ground truth for this
port: they were produced by the exact script logic this notebook just re-implemented, run
against the same raw files. Joining the new tables to the old ones on `poll_id` and comparing
`number_of_votes`, `number_of_electors`, `proportion_of_turnout`, `poll_total_candidate_votes`,
and every party-vote column should show max-abs-differences of (numerically) zero -- this is a
straight port, not a re-derivation, so any nonzero difference means something in the port
diverged from the original and needs to be investigated rather than waved away.


In [9]:
def verify_election(election_id, new_df, party_fields, gt_relpath):
    gt = pd.read_csv(GROUND_TRUTH / gt_relpath, dtype={"poll_id": str}, low_memory=False)
    merged = new_df.merge(gt, on="poll_id", suffixes=("_new", "_gt"), how="outer", indicator=True)

    only_new = int((merged["_merge"] == "left_only").sum())
    only_gt = int((merged["_merge"] == "right_only").sum())

    compare_cols = ["number_of_votes", "number_of_electors", "proportion_of_turnout", "poll_total_candidate_votes"] + party_fields
    diffs = {}
    for col in compare_cols:
        a = pd.to_numeric(merged.get(f"{col}_new"), errors="coerce")
        b = pd.to_numeric(merged.get(f"{col}_gt"), errors="coerce")
        diffs[col] = float((a - b).abs().max())

    max_diff = max(diffs.values())
    passed = only_new == 0 and only_gt == 0 and max_diff < 1e-6
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {election_id}: {len(new_df)} rows (new) vs {len(gt)} rows (ground truth); "
          f"unmatched new={only_new}, unmatched ground truth={only_gt}; max abs diff across "
          f"{len(compare_cols)} numeric columns = {max_diff:.3g}")
    if not passed:
        worst = sorted(diffs.items(), key=lambda kv: -kv[1])[:5]
        print("  largest per-column diffs:", worst)
    return passed


results = {
    "municipal_2023_mayor": verify_election(
        "municipal_2023_mayor", municipal_df, municipal_party_fields,
        "municipal_2023_mayor/turnout/toronto_municipal_2023_mayor_turnout_subdivisions.csv",
    ),
    "provincial_2025": verify_election(
        "provincial_2025", provincial_df, provincial_party_fields,
        "provincial_2025/turnout/toronto_provincial_2025_turnout_poll_divisions.csv",
    ),
    "federal_2025": verify_election(
        "federal_2025", federal_df, federal_party_fields,
        "federal_2025/turnout/toronto_federal_2025_turnout_poll_divisions.csv",
    ),
}

print()
if all(results.values()):
    print("ALL ELECTIONS PASS -- normalized output matches the archived pipeline exactly.")
else:
    failed = [k for k, v in results.items() if not v]
    raise AssertionError(f"Verification failed for: {failed}")


[PASS] municipal_2023_mayor: 1545 rows (new) vs 1545 rows (ground truth); unmatched new=0, unmatched ground truth=0; max abs diff across 5 numeric columns = 1.11e-16
[PASS] provincial_2025: 1532 rows (new) vs 1532 rows (ground truth); unmatched new=0, unmatched ground truth=0; max abs diff across 19 numeric columns = 1.11e-16


[PASS] federal_2025: 5069 rows (new) vs 5069 rows (ground truth); unmatched new=0, unmatched ground truth=0; max abs diff across 17 numeric columns = 8.33e-17

ALL ELECTIONS PASS -- normalized output matches the archived pipeline exactly.


## Takeaways for the next notebook

All three elections now live in one schema, poll-keyed, with candidate/party detail folded
in and geometry preserved in a matching GeoJSON. The next notebook in the pipeline
(`03_interpolate_votes_to_tracts.ipynb`, per the rebuild plan) needs exactly this shape: it
will area/population-weight-allocate each poll's `number_of_votes`/`number_of_electors` (and
the Block-4 competitiveness features derived from the party columns) onto census tracts using
these polygons. Two properties of this output make that step tractable:

- Every row -- including combined/special/no-geometry rows -- carries a coherent
  `number_of_electors`, so a census-tract interpolation that sums electors across polls will
  not silently lose population just because a poll's polygon is missing or its votes were
  folded into another poll's total.
- `vote_in_other_division` and `source_note` stay attached to each row, so a later reader (or
  a spatial join that behaves unexpectedly for one riding) can trace *why* a given poll's
  numbers look the way they do without re-reading the archived scripts.
